# IPL Data Analysis (2008–2024)
### Indian Premier League — End-to-End Data Analysis Project

This notebook analyses **IPL match and delivery data (1,095 matches · 260,920 deliveries)**.  
It covers the full pipeline: data loading → cleaning → analysis → visualisation → business insights.

**Tools used:** Python · Pandas · Matplotlib

---

## Section 1 — Data Loading

In [ ]:
import pandas as pd

# Load both IPL datasets
matches    = pd.read_csv("/content/matches.csv")
deliveries = pd.read_csv("/content/deliveries.csv")

print("Matches shape   :", matches.shape)
print("Deliveries shape:", deliveries.shape)

In [ ]:
# Quick look at matches table
matches.head()

In [ ]:
# Quick look at deliveries table
deliveries.head()

In [ ]:
# Check column names
print("Matches columns:")
print(list(matches.columns))

print("\nDeliveries columns:")
print(list(deliveries.columns))

## Section 2 — Data Cleaning

In [ ]:
# Check missing values in matches
matches.isnull().sum()

In [ ]:
# Check missing values in deliveries
deliveries.isnull().sum()

In [ ]:
# Drop columns not needed for analysis
matches = matches.drop(columns=[
    "umpire1",       # Not relevant for performance analysis
    "umpire2",       # Not relevant for performance analysis
    "method",        # Mostly null - DLS method column
    "target_runs",   # Not needed
    "target_overs"   # Not needed
])

print("Cleaned matches shape:", matches.shape)

In [ ]:
# Fill missing winner with 'No Result' — some matches were abandoned
matches["winner"] = matches["winner"].fillna("No Result")

# Fill missing city with 'Unknown'
matches["city"] = matches["city"].fillna("Unknown")

print("Missing values after cleaning:")
print(matches.isnull().sum())

In [ ]:
# Merge deliveries with matches to get season and venue info
df = deliveries.merge(
    matches[["id", "season", "venue", "city"]],
    left_on="match_id",
    right_on="id",
    how="left"
)

print("Merged dataframe shape:", df.shape)

In [ ]:
# Final check — confirm no major missing values
df.isnull().sum()

In [ ]:
# Save cleaned files for reuse
matches.to_csv("cleaned_matches.csv", index=False)
df.to_csv("cleaned_deliveries.csv", index=False)
print("Cleaned files saved.")

## Section 3 — Analysis

In [ ]:
import pandas as pd

# Reload cleaned datasets
matches    = pd.read_csv("cleaned_matches.csv")
df         = pd.read_csv("cleaned_deliveries.csv")

print("Matches shape   :", matches.shape)
print("Deliveries shape:", df.shape)

### 3.1 — Most Successful Teams

In [ ]:
# Total wins per team (exclude No Result matches)
team_wins = (
    matches[matches["winner"] != "No Result"]
    .groupby("winner")["id"]
    .count()
    .sort_values(ascending=False)
    .head(10)
)

print("===== TOP 10 TEAMS BY WINS =====")
print(team_wins)

**Business Insight:**
- Mumbai Indians lead IPL with the most wins — consistent squad building and retention of match-winners is the main reason.
- Chennai Super Kings are a close second despite being banned for 2 seasons — shows how strong their core system is.
- Win count alone doesn't tell the full story — teams that played more seasons naturally have more wins.

### 3.2 — Top Run Scorers (All Time)

In [ ]:
# Total runs scored by each batsman across all seasons
top_batsmen = (
    df.groupby("batter")["batsman_runs"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print("===== TOP 10 ALL-TIME RUN SCORERS =====")
print(top_batsmen)

**Business Insight:**
- Virat Kohli leads IPL run charts by a significant margin — 15+ seasons of consistent opening partnerships.
- Top scorers are almost always openers or No.3 batsmen — they face the most deliveries per match.
- High run scorers are the most expensive players at IPL auctions — franchises pay a premium for reliability.

### 3.3 — Top Wicket Takers (All Time)

In [ ]:
# Count only bowler-credited wickets — exclude run outs
wickets = df[
    (df["is_wicket"] == 1) &
    (df["dismissal_kind"] != "run out")
]

top_bowlers = (
    wickets.groupby("bowler")["is_wicket"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print("===== TOP 10 ALL-TIME WICKET TAKERS =====")
print(top_bowlers)

**Business Insight:**
- Dwayne Bravo and Lasith Malinga dominate — both are death-over specialists, the most crucial phase in T20.
- Wicket-taking ability is rarer and more valuable than economy rate alone in T20 cricket.
- Top bowlers on this list played for successful franchises — good teams attract and retain elite bowlers.

### 3.4 — Toss Impact on Match Result

In [ ]:
# Check if toss winner also won the match
matches["toss_won_match"] = (matches["toss_winner"] == matches["winner"])

toss_count = matches["toss_won_match"].value_counts()
toss_pct   = (toss_count / toss_count.sum() * 100).round(2)

print("===== TOSS IMPACT ON RESULT =====")
print(pd.DataFrame({"Count": toss_count, "Percentage": toss_pct}))

In [ ]:
# Field vs Bat decision after winning toss
toss_decision = (
    matches.groupby("toss_decision")["toss_won_match"]
    .value_counts()
    .unstack(fill_value=0)
)

print("\n===== TOSS DECISION IMPACT =====")
print(toss_decision)

**Business Insight:**
- Toss winner wins only ~52% of matches — barely better than a coin flip, so toss is overrated.
- Teams choosing to field first after winning toss win slightly more — chasing is psychologically easier in T20.
- The real advantage is pitch reading and adjusting strategy — not just the toss itself.

### 3.5 — Venue Analysis

In [ ]:
# Most matches hosted per venue
venue_matches = (
    matches.groupby("venue")["id"]
    .count()
    .sort_values(ascending=False)
    .head(10)
)

print("===== TOP 10 VENUES BY MATCHES HOSTED =====")
print(venue_matches)

In [ ]:
# Highest scoring venues — avg total runs per match
venue_runs = (
    df.groupby(["match_id", "venue"])["total_runs"]
    .sum()
    .reset_index()
    .groupby("venue")["total_runs"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .round(2)
)

print("\n===== TOP 10 HIGHEST SCORING VENUES (Avg Runs/Match) =====")
print(venue_runs)

**Business Insight:**
- Wankhede Stadium and M. Chinnaswamy are notoriously high-scoring — small boundaries and flat pitches favour batsmen.
- Venues with the most matches hosted are home grounds of successful franchises — home crowd advantage is real.
- Bowlers at high-scoring venues need different game plans — yorkers and slower balls over pace.

### 3.6 — Season-wise Run Trends

In [ ]:
# Total runs scored per season
season_runs = (
    df.groupby("season")["total_runs"]
    .sum()
    .sort_index()
)

print("===== SEASON-WISE TOTAL RUNS =====")
print(season_runs)

In [ ]:
# Average runs per match per season — cleaner metric
season_matches = matches.groupby("season")["id"].count()
season_avg     = (season_runs / season_matches).round(2)

print("\n===== AVG RUNS PER MATCH PER SEASON =====")
print(season_avg)

**Business Insight:**
- Total runs have increased steadily since 2008 — better bats, evolved batting techniques, and boundary fielding restrictions.
- Average runs per match is the cleaner metric — it removes the effect of extra matches in some seasons.
- 2011–2013 saw a dip due to slower pitches prepared to counter power-hitting — shows strategic adaptation.

### 3.7 — Top Individual Innings in a Single Match

In [ ]:
# Highest individual scores in a single match innings
top_innings = (
    df.groupby(["match_id", "batter"])["batsman_runs"]
    .sum()
    .reset_index()
    .sort_values("batsman_runs", ascending=False)
    .head(10)
)

print("===== TOP 10 INDIVIDUAL INNINGS =====")
print(top_innings)

**Business Insight:**
- Individual brilliance can single-handedly win T20 matches — unlike Test cricket where team effort dominates.
- Most top innings are scored while chasing — pressure situations bring out extraordinary performances.
- These performances almost always result in Player of the Match awards and massive auction value next season.

## Section 4 — Visualisation

> All 7 plots use Matplotlib only. Each chart is saved as a `.png` file for use in Power BI and GitHub README.

In [ ]:
import matplotlib.pyplot as plt

# Reload datasets
matches    = pd.read_csv("cleaned_matches.csv")
df         = pd.read_csv("cleaned_deliveries.csv")

### Plot 1 — Top 10 Teams by Wins

In [ ]:
# --- Plot 1: Top 10 Teams by Total Wins ---
team_wins = (
    matches[matches["winner"] != "No Result"]
    .groupby("winner")["id"]
    .count()
    .sort_values(ascending=False)
    .head(10)
)

plt.figure(figsize=(12, 5))
team_wins.plot(kind="bar", color="steelblue")

plt.title("Top 10 IPL Teams by Total Wins", fontsize=14, fontweight="bold")
plt.xlabel("Team", fontsize=11)
plt.ylabel("Total Wins", fontsize=11)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("plot1_team_wins.png", dpi=150, bbox_inches="tight")
plt.show()

### Plot 2 — Top 10 Run Scorers

In [ ]:
# --- Plot 2: Top 10 All-Time Run Scorers ---
top_batsmen = (
    df.groupby("batter")["batsman_runs"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

plt.figure(figsize=(12, 5))
top_batsmen.plot(kind="bar", color="coral")

plt.title("Top 10 IPL Run Scorers (All Time)", fontsize=14, fontweight="bold")
plt.xlabel("Batsman", fontsize=11)
plt.ylabel("Total Runs", fontsize=11)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("plot2_top_batsmen.png", dpi=150, bbox_inches="tight")
plt.show()

### Plot 3 — Top 10 Wicket Takers

In [ ]:
# --- Plot 3: Top 10 All-Time Wicket Takers ---
wickets = df[
    (df["is_wicket"] == 1) &
    (df["dismissal_kind"] != "run out")
]

top_bowlers = (
    wickets.groupby("bowler")["is_wicket"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

plt.figure(figsize=(12, 5))
top_bowlers.plot(kind="bar", color="mediumseagreen")

plt.title("Top 10 IPL Wicket Takers (All Time)", fontsize=14, fontweight="bold")
plt.xlabel("Bowler", fontsize=11)
plt.ylabel("Total Wickets", fontsize=11)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("plot3_top_bowlers.png", dpi=150, bbox_inches="tight")
plt.show()

### Plot 4 — Toss Impact on Result

In [ ]:
# --- Plot 4: Toss Impact (Pie Chart) ---
matches["toss_won_match"] = (matches["toss_winner"] == matches["winner"])
toss_count = matches["toss_won_match"].value_counts()

plt.figure(figsize=(7, 5))
plt.pie(
    toss_count,
    labels=["Toss Winner Won", "Toss Winner Lost"],
    autopct="%1.1f%%",
    colors=["#2ecc71", "#e74c3c"],
    startangle=90
)
plt.title("Did Toss Winner Win the Match?", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("plot4_toss_impact.png", dpi=150, bbox_inches="tight")
plt.show()

### Plot 5 — Top Venues by Matches Hosted

In [ ]:
# --- Plot 5: Top 10 Venues by Matches Hosted ---
venue_matches = (
    matches.groupby("venue")["id"]
    .count()
    .sort_values(ascending=False)
    .head(10)
)

plt.figure(figsize=(12, 5))
venue_matches.plot(kind="bar", color="mediumpurple")

plt.title("Top 10 Venues by Matches Hosted", fontsize=14, fontweight="bold")
plt.xlabel("Venue", fontsize=11)
plt.ylabel("Number of Matches", fontsize=11)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("plot5_venues.png", dpi=150, bbox_inches="tight")
plt.show()

### Plot 6 — Season-wise Run Trend

In [ ]:
# --- Plot 6: Season-wise Total Runs ---
season_runs = (
    df.groupby("season")["total_runs"]
    .sum()
    .sort_index()
)

plt.figure(figsize=(12, 5))
season_runs.plot(kind="line", marker="o", color="steelblue")

plt.title("Season-wise Total Runs Scored in IPL", fontsize=14, fontweight="bold")
plt.xlabel("Season", fontsize=11)
plt.ylabel("Total Runs", fontsize=11)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("plot6_season_runs.png", dpi=150, bbox_inches="tight")
plt.show()

### Plot 7 — Highest Scoring Venues

In [ ]:
# --- Plot 7: Highest Scoring Venues (Avg Runs per Match) ---
venue_runs = (
    df.groupby(["match_id", "venue"])["total_runs"]
    .sum()
    .reset_index()
    .groupby("venue")["total_runs"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .round(2)
)

plt.figure(figsize=(12, 5))
venue_runs.plot(kind="bar", color="tomato")

plt.title("Top 10 Highest Scoring Venues (Avg Runs/Match)", fontsize=14, fontweight="bold")
plt.xlabel("Venue", fontsize=11)
plt.ylabel("Avg Runs per Match", fontsize=11)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("plot7_venue_runs.png", dpi=150, bbox_inches="tight")
plt.show()

## Section 5 — Final Summary & Key Findings

| # | Finding | Business Insight |
|---|---------|-----------------|
| 1 | Mumbai Indians have the most IPL wins | Consistent squad retention = sustained success over seasons |
| 2 | Virat Kohli is the all-time top scorer | Opener consistency is the biggest driver of team totals |
| 3 | Bravo and Malinga lead wicket charts | Death-over specialists are the most valuable bowlers in T20 |
| 4 | Toss winner wins only ~52% of matches | Toss advantage is marginal — execution matters far more |
| 5 | Wankhede is highest scoring venue | Pitch and boundary size directly shape team strategy |
| 6 | Runs per match increasing every season | T20 batting has evolved significantly since IPL started in 2008 |
| 7 | Individual innings can reach 170+ runs | One player can completely change the outcome of a T20 match |

---
*Dataset: IPL Complete Dataset (2008–2024) — publicly available on Kaggle.*